# Projection pipeline audit

## tl;dr

- DraftKings and FanDuel overlap closely, while Kalshi-involved published rows trigger disagreement flags 44.3% of the time.
- Kalshi updated historical dispersion for 0 stats; 57 source curves were quarantined.
- 135 of 158 eligible Kalshi curves had no 24-hour trades.
- All 171 fantasy rows have blank player names and none is marked complete.

## Context & Methods

This diagnostic notebook audits run `20260826T152136Z-829c0109` at player/stat/source grain. Source means are compared only where both sources passed model validation. Kalshi liquidity is measured on the exact quote rows that contributed to passed source curves.

### Key Assumptions

- Pairwise percentage gaps use the symmetric absolute difference.
- The two sportsbooks form the comparison baseline for three-source overlaps; this is a consistency check, not proof that the books are correct.
- Cumulative volume, open interest, 24-hour volume, spread, and top-of-book size describe different aspects of liquidity and are not interchangeable.

In [1]:
from pathlib import Path
import pandas as pd
from analysis import RUN_DIR, compute_audit
results = compute_audit()
print(f'Run directory: {RUN_DIR}')
print(f"Published consensus rows: {results['summary']['published_consensus_rows']}")


Run directory: /home/john/ff-market-projections/runs/20260826T152136Z-829c0109
Published consensus rows: 356


## Data

The analysis reads the run-scoped priced markets, source projections, consensus statistics, fantasy projections, model validation report, manifest, and raw Kalshi order-book snapshot.

In [2]:
print(pd.Series(results['summary'], name='value').to_string())


run_id                                                             20260826T152136Z-829c0109
source_skew_seconds                                                                 5.218714
eligible_source_curves                     {'draftkings': 294, 'fanduel': 136, 'kalshi': ...
published_consensus_rows                                                                 356
single_source_consensus_rows                                                             166
single_source_consensus_rate                                                        0.466292
kalshi_disagreement_rows                                                                  70
kalshi_disagreement_rate                                                            0.443038
non_kalshi_disagreement_rows                                                              10
non_kalshi_disagreement_rate                                                        0.050505
kalshi_raw_quotes                                                     

## Results

### Pairwise source agreement

In [3]:
print(pd.DataFrame(results['pair_summary']).to_string(index=False))


                 pair  overlap  correlation  median_abs_difference  mean_signed_difference
DraftKings vs FanDuel      127     0.999048               0.026297               -0.008981
 DraftKings vs Kalshi      100     0.984996               0.240212               -0.132577
    FanDuel vs Kalshi       47     0.981138               0.310147               -0.158501


### Three-source behavior by stat

In [4]:
print(pd.DataFrame(results['triple_by_stat']).to_string(index=False))


        stat_label  overlaps  median_kalshi_vs_books  median_abs_kalshi_vs_books  median_consensus_shift disagreement_flags
Passing Touchdowns         1               -0.247748                    0.247748               -0.082583               True
     Passing Yards         3               -0.288909                    0.288909               -0.096303                  3
   Receiving Yards        13               -0.144170                    0.144170               -0.048057                  9
Rushing Touchdowns        11                0.137898                    0.262370                0.045966                  8
     Rushing Yards        14               -0.316696                    0.316696               -0.105565                 14


### Kalshi liquidity funnel and largest disagreements

In [5]:
print(pd.DataFrame(results['kalshi_funnel']).to_string(index=False))
print('\nLiquidity correlations with absolute Kalshi/book disagreement:')
print(pd.DataFrame(results['liquidity_correlations']).T.to_string())
print('\nLargest disagreements:')
print(pd.DataFrame(results['triple_outliers']).to_string(index=False))


                              stage  count
               Raw Kalshi contracts   1129
      Two-sided and spread-eligible    336
              Quotes on curves used    187
        Eligible player/stat curves    158
Stats with Kalshi dispersion update      0

Liquidity correlations with absolute Kalshi/book disagreement:
                            rho   p_value     n
total_volume          -0.473701  0.001531  42.0
minimum_open_interest -0.477090  0.001400  42.0
volume_24h            -0.150949  0.339972  42.0
median_spread          0.064738  0.683778  42.0

Largest disagreements:
canonical_player_name         stat_label  draftkings_mean  fanduel_mean  kalshi_mean  kalshi_vs_books  consensus_vs_books  contributing_quotes  total_volume  minimum_open_interest  volume_24h  median_spread  minimum_top_size
           James Cook Rushing Touchdowns        12.156919     12.184585    21.893103         0.798829            0.266276                    1        119.72                 119.72        0.

### Fantasy output reasonableness

In [6]:
print(pd.DataFrame(results['top_scores']).to_string(index=False))


canonical_player_name canonical_position   scoring_profile  fpts_standard  fpts_full_ppr  passing_yards  passing_touchdowns  rushing_yards  rushing_touchdowns  receiving_yards  receiving_touchdowns  receptions
           Josh Allen                 QB   passing|rushing     450.196316     450.196316    4740.232892           29.533652     574.214382           14.171826              NaN                   NaN         NaN
           Drake Maye                 QB   passing|rushing     413.174183     413.174183    5089.679801           32.303199     530.441870            4.555001              NaN                   NaN         NaN
          Jalen Hurts                 QB   passing|rushing     391.729678     391.729678    4198.984178           26.640012     530.441870           10.694346              NaN                   NaN         NaN
        Lamar Jackson                 QB   passing|rushing     378.674179     378.674179    4231.787131           28.425369     683.680166            4.555534  

## Takeaways

1. Kalshi is useful as a third signal, but the current equal weighting is not supported by the observed liquidity and cross-source disagreement.
2. The dispersion-update fallback and curve quarantine are working as safety mechanisms.
3. The final fantasy output needs a blocking validation for player names and clearer treatment of incomplete scores.
4. Source-specific inversion should be validated against historical market snapshots and realized outcomes before the resulting means are treated as conventional fantasy projections.